# Code Mode Agents - Data Analysis

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/code-mode-analysis/code-mode-analysis-tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/code-mode-analysis
    !uv pip install -r requirements.txt
    !uv pip install keyrings.alt pygments
    !mkdir -p ~/.config/python_keyring && echo -e '[backend]\ndefault-keyring=keyrings.alt.file.PlaintextKeyring' > ~/.config/python_keyring/keyringrc.cfg
    %env TERM=dumb

from utils.file_viewer import view_file

In [ ]:
# Set the Anthropic key. Skip this if it's already in a .env or your environment —
# config.py calls load_dotenv() for you.
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")

In [ ]:
# set Flyte cluster

Runing Flyte workflows

--local
--tui

Local runs still get caching, retries, TUI

remote


Fanout
scale
containerized
UI


### 0. Download the dataset (optional, but do it before a workshop)

In [ ]:
!flyte run --local step0_download_data.py download

### 1. The sandbox, with no LLM anywhere

In [ ]:

!flyte run --local step1_sandbox.py monthly_tip_trend

### 2. The model writes the program

In [ ]:
!flyte run --local step2_generated_code.py analyze \
    --question "Did tipping change between January and December 2024?"

In [ ]:
# The task was decorated @env.task(report=True), so it wrote an HTML report.
# Render it here — charts, metrics, and the program the model actually wrote.
from report import show_latest

show_latest()

### 3. The agent, and the fan-out

`Agent(code_mode=True)` writes one program, and `query` is an `@env.task` — so every query the model writes dispatches as a **durable child task**. Look for `flyte_map("query", sqls, months, concurrency=4)` in the program it wrote below.

> **Note:** running `--local`, that `flyte_map` executes **sequentially** — Flyte warns you so in the output. The generated code and the answer are identical, but the twelve-parallel-containers payoff only happens on a cluster. Run this one remotely (drop `--local`) and open the run in the UI to see the child tasks.

In [ ]:
!flyte run --local step3_agent_report.py analyze \
    --question "Rank the boroughs by tip rate and show how it moved through 2024"

In [ ]:
from report import show_latest

show_latest()

### 4. Why bother? Measure it.

In [ ]:
!flyte run --local step4_compare_modes.py compare \
    --question "Which borough tipped best in each quarter of 2024?"

In [ ]:
# Turns and tokens, side by side: sequential tool calling vs code mode.
from report import show_latest

show_latest()

### 5. Serve it as a chat app (stretch)

Two front ends, and the difference is the point of step 3.

**Here in the notebook** — a small Gradio UI over the same agent. Gradio renders inline in Colab, which is why we use it for the notebook. The agent runs in this process, so there are **no durable child tasks and no real fan-out**. It's the right way to poke at the prompt.

**Deployed** (last cell) — the native `AgentChatAppEnvironment`: the chat UI, streaming, and the endpoint in one declaration, and every message becomes a **durable Flyte run** whose `query` calls fan out as child tasks you can click into.

Try:
- *"Do riders in Brooklyn and the Bronx really tip less than Manhattan, or is something else going on?"*
- *"How did the tip rate move month by month through 2024? Chart it."*

In [ ]:
!pip install -q gradio

In [ ]:
# A Gradio front end over the same agent — Gradio renders inline in Colab, which is
# why we use it here rather than the native chat app (that one is what gets deployed).
#
# The agent runs in this process, so there are no durable child tasks and no real
# fan-out. Deploy it (next cell) for that.
import asyncio

import gradio as gr

import flyte
import tools
from analysis import build_agent

flyte.init()  # local: tasks run in-process


def ask(question):
    tools.new_report()
    agent, usage = build_agent(code_mode=True)
    result = asyncio.run(agent.run.aio(question))

    program = usage.programs[0] if usage.programs else (result.code or "")
    charts = "".join(tools.collect_report())
    return result.summary or result.error, charts, program


with gr.Blocks(title="NYC Taxi analyst") as demo:
    gr.Markdown(
        "## NYC Taxi analyst\n"
        "Ask a question. Claude writes one Python program, the Monty sandbox runs it, "
        "and the only things it can touch are the tools we registered."
    )
    question = gr.Textbox(
        label="Question",
        placeholder="How did the tip rate move month by month through 2024?",
    )
    ask_btn = gr.Button("Ask", variant="primary")

    answer = gr.Markdown(label="Answer")
    report = gr.HTML(label="Report")
    program = gr.Code(label="The program the model wrote", language="python")

    ask_btn.click(ask, inputs=question, outputs=[answer, report, program])
    question.submit(ask, inputs=question, outputs=[answer, report, program])

    gr.Examples(
        examples=[
            "Do riders in Brooklyn and the Bronx really tip less than Manhattan, "
            "or is something else going on?",
            "How did the tip rate move month by month through 2024? Chart it.",
            "How do JFK and LaGuardia airport pickups compare on trip distance, fare, "
            "and tip rate across 2024?",
        ],
        inputs=question,
    )

demo.launch(share=True)  # Colab renders this inline; share=True also gives a public link

In [ ]:
# Deploy it to the cluster instead — needs a Flyte/Union connection.
!python step5_chat_app.py deploy